# HWP (한글) 문서 로더

한글(HWP)은 한글과컴퓨터에서 개발한 워드프로세서로, 한국에서 가장 널리 사용되는 문서 형식입니다.

공공기관, 기업, 학교 등에서 광범위하게 사용되기 때문에 한국 개발자라면 HWP 파일을 처리해야 하는 경우가 많습니다.


# 02. HWP(한글) 문서 로더

## 학습 목표
- HWP 파일의 내부 구조(OLE 복합 문서 형식)를 이해해요
- `UnstructuredFileLoader`와 커스텀 로더 두 가지 방법으로 HWP를 로딩해요
- 상황에 맞는 HWP 처리 전략을 선택할 수 있어요

## 사전 지식
- 00-Document-Loader 노트북 완료
- `pip install unstructured olefile` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — 이번에는 한국에서 널리 쓰이는 **HWP 형식**을 처리해요.

```mermaid
flowchart LR
    A[📄 Load<br/>HWP 문서]:::current --> B[✂️ Chunk]:::process
    B --> C[🔢 Embed]:::process
    C --> D[🗄️ Store]:::storage
    D --> E[🔍 Retrieve]:::process
    E --> F[💬 Generate]:::output

    classDef current fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```


## 학습 목표

- HWP 파일의 특징과 구조 이해
- HWP 파일을 Document 객체로 변환하는 방법 학습
- HWP 로더 직접 구현하기


## HWP 파일 특징

### HWP 파일이란?

- **파일 확장자**: `.hwp`
- **개발사**: 한글과컴퓨터
- **용도**: 워드프로세싱 (문서 작성, 편집)
- **특징**: 한글 처리에 최적화, 복잡한 레이아웃 지원

### HWP 파일 구조

HWP 파일은 내부적으로 **복합 문서(Compound File Binary Format)** 구조를 사용합니다.

- OLE(Object Linking and Embedding) 형식
- 여러 스트림(stream)으로 구성
- 텍스트, 이미지, 서식 등이 분리되어 저장


> 🎯 **강의 포인트**: HWP는 LangChain 공식 로더가 없습니다. 한국 공공기관 문서를 처리하는 프로젝트라면 반드시 만나게 되는 형식이므로, 커스텀 로더 구현 방법을 꼭 익혀두세요.

## HWP 로더 직접 구현하기

LangChain에는 공식 HWP 로더가 아직 포함되어 있지 않으므로, 직접 구현해야 합니다.

HWP 파일을 파싱하는 방법은 여러 가지가 있습니다:

### 방법 1: Unstructured 라이브러리 사용
- 가장 간단하고 권장되는 방법
- 다양한 문서 형식을 통합 지원

### 방법 2: olefile + 직접 파싱
- HWP 파일 구조에 직접 접근
- 세밀한 제어 가능하지만 복잡함

### 방법 3: hwp5 라이브러리
- HWP 5.0 형식 전용
- 구조화된 접근 가능

이 노트북에서는 **Unstructured 라이브러리**를 사용하는 방법을 다룹니다.


In [1]:
# 환경 설정
from dotenv import load_dotenv

load_dotenv()

# HWP 파일 경로
FILE_PATH = "./data/디지털 정부혁신 추진계획.hwp"


## 커스텀 HWP 로더 클래스 구현

직접 HWP 로더를 구현할 수 있습니다.

**필요한 패키지**:
```bash
pip install olefile zlib
```


> 🔑 **핵심 개념**: `BaseLoader`를 상속해서 `load()` 메서드만 구현하면 어떤 형식이든 LangChain Document로 변환할 수 있습니다. 커스텀 로더 패턴은 사내 특수 형식 처리에 매우 유용합니다.

In [6]:
from typing import Any, Dict, List, Optional, Iterator
import olefile
import zlib
import struct
import re
import unicodedata
from langchain.schema import Document
from langchain.document_loaders.base import BaseLoader


class HWPLoader(BaseLoader):
    """HWP 파일 읽기 클래스. HWP 파일의 내용을 읽습니다."""

    def __init__(self, file_path: str, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.file_path = file_path
        self.extra_info = {"source": file_path}
        self._initialize_constants()

    def _initialize_constants(self) -> None:
        """상수 초기화 메서드"""
        self.FILE_HEADER_SECTION = "FileHeader"
        self.HWP_SUMMARY_SECTION = "\x05HwpSummaryInformation"
        self.SECTION_NAME_LENGTH = len("Section")
        self.BODYTEXT_SECTION = "BodyText"
        self.HWP_TEXT_TAGS = [67]

    def lazy_load(self) -> Iterator[Document]:
        """HWP 파일에서 데이터를 로드하고 표를 추출합니다.

        Yields:
            Document: 추출된 문서
        """
        load_file = olefile.OleFileIO(self.file_path)
        file_dir = load_file.listdir()

        if not self._is_valid_hwp(file_dir):
            raise ValueError("유효하지 않은 HWP 파일입니다.")

        result_text = self._extract_text(load_file, file_dir)
        yield self._create_document(text=result_text, extra_info=self.extra_info)

    def _is_valid_hwp(self, dirs: List[List[str]]) -> bool:
        """HWP 파일의 유효성을 검사합니다."""
        return [self.FILE_HEADER_SECTION] in dirs and [self.HWP_SUMMARY_SECTION] in dirs

    def _get_body_sections(self, dirs: List[List[str]]) -> List[str]:
        """본문 섹션 목록을 반환합니다."""
        section_numbers = [
            int(d[1][self.SECTION_NAME_LENGTH :])
            for d in dirs
            if d[0] == self.BODYTEXT_SECTION
        ]
        return [
            f"{self.BODYTEXT_SECTION}/Section{num}" for num in sorted(section_numbers)
        ]

    def _create_document(
        self, text: str, extra_info: Optional[Dict] = None
    ) -> Document:
        """문서 객체를 생성합니다."""
        return Document(page_content=text, metadata=extra_info or {})

    def _extract_text(
        self, load_file: olefile.OleFileIO, file_dir: List[List[str]]
    ) -> str:
        """모든 섹션에서 텍스트를 추출합니다."""
        sections = self._get_body_sections(file_dir)
        return "\n".join(
            self._get_text_from_section(load_file, section) for section in sections
        )

    def _is_compressed(self, load_file: olefile.OleFileIO) -> bool:
        """파일이 압축되었는지 확인합니다."""
        with load_file.openstream(self.FILE_HEADER_SECTION) as header:
            header_data = header.read()
            return bool(header_data[36] & 1)

    def _get_text_from_section(self, load_file: olefile.OleFileIO, section: str) -> str:
        """특정 섹션에서 텍스트를 추출합니다."""
        with load_file.openstream(section) as bodytext:
            data = bodytext.read()

        unpacked_data = (
            zlib.decompress(data, -15) if self._is_compressed(load_file) else data
        )

        text = []
        i = 0
        while i < len(unpacked_data):
            header, rec_type, rec_len = self._parse_record_header(
                unpacked_data[i : i + 4]
            )
            if rec_type in self.HWP_TEXT_TAGS:
                rec_data = unpacked_data[i + 4 : i + 4 + rec_len]
                text.append(rec_data.decode("utf-16"))
            i += 4 + rec_len

        text = "\n".join(text)
        text = self.remove_chinese_characters(text)
        text = self.remove_control_characters(text)
        return text

    @staticmethod
    def remove_chinese_characters(s: str):
        """중국어 문자를 제거합니다."""
        return re.sub(r"[\u4e00-\u9fff]+", "", s)

    @staticmethod
    def remove_control_characters(s):
        """깨지는 문자 제거"""
        return "".join(ch for ch in s if unicodedata.category(ch)[0] != "C")

    @staticmethod
    def _parse_record_header(header_bytes: bytes) -> tuple:
        """레코드 헤더를 파싱합니다."""
        header = struct.unpack_from("<I", header_bytes)[0]
        rec_type = header & 0x3FF
        rec_len = (header >> 20) & 0xFFF
        return header, rec_type, rec_len



print("✅ CustomHWPLoader 클래스 정의 완료")

✅ CustomHWPLoader 클래스 정의 완료


> ⚠️ **자주 하는 실수**: HWP 파일 인코딩은 `utf-16-le`, `cp949`, `euc-kr` 순으로 시도해야 합니다. `utf-8`로 먼저 시도하면 한글이 깨지거나 빈 결과가 나올 수 있습니다.

> 💡 **실무 팁**: HWP 처리가 불안정하다면 한글과컴퓨터 공식 API나 LibreOffice를 이용해 HWP → DOCX 변환 후 Word 로더로 처리하는 방법이 더 안정적입니다.

In [7]:
# 커스텀 HWP 로더 사용
custom_loader = HWPLoader(FILE_PATH)

print("📄 커스텀 HWP 로더로 파일 로딩 중...")
try:
    custom_docs = custom_loader.load()
    
    print("=" * 60)
    print("✅ 로딩 완료 (커스텀 로더)")
    print("=" * 60)
    print(f"Document 개수: {len(custom_docs)}")
    print(f"\n내용 미리보기 (처음 500자):")
    print(custom_docs[0].page_content[:500])
    print("...")
except Exception as e:
    print(f"⚠️ 로딩 실패: {e}")
    print("💡 Unstructured 라이브러리 사용을 권장합니다.")


📄 커스텀 HWP 로더로 파일 로딩 중...
✅ 로딩 완료 (커스텀 로더)
Document 개수: 1

내용 미리보기 (처음 500자):
디지털 정부혁신 추진계획2019. 10. 29.      관계부처 합동순    서Ⅰ. 개요ȃ 1Ⅱ. 디지털 정부혁신 추진계획ㆬȃ 2  1. 우선 추진과제ȃ 2     ① 선제적·통합적 대국민 서비스 혁신     ② 공공부문 마이데이터 활성화     ③ 시민참여를 위한 플랫폼 고도화     ④ 현장중심 협업을 지원하는 스마트 업무환경 구현     ⑤ 클라우드와 디지털서비스 이용 활성화     ⑥ 개방형 데이터·서비스 생태계 구축  2. 중장기 범정부 디지털 전환 로드맵 수립ᲈȃ 4Ⅲ. 추진체계 및 일정ȃ 4 [붙임] 디지털 정부혁신 우선 추진과제(상세)ᬜȃ 8Ⅰ. 개 요□ 추진 배경 ○ 우리나라는 국가적 초고속 정보통신망 투자와 적극적인 공공정보화 사업 추진에 힘입어 세계 최고수준의 전자정부를 구축‧운영     * UN전자정부평가에서 2010‧12‧14년 1위, 16‧18년 3위, UN공공행정상 13회 수상 ○ 그러나, 인공지능‧클라우드 중심의 디지털 전환(Digital Transfo
...


## HWP 파일 처리 시 주의사항

### 1. 버전 호환성

HWP 파일은 여러 버전이 있습니다:
- **HWP 3.0 이하**: 구형 형식, 파싱이 복잡
- **HWP 5.0+**: OLE 형식, 상대적으로 파싱 용이
- **HWPX**: XML 기반 형식, 압축된 XML 파일

### 2. 인코딩 문제

HWP 파일은 한글 처리를 위해 다양한 인코딩을 사용:
- CP949 (Windows 한글 코드 페이지)
- EUC-KR
- UTF-16LE

### 3. 복잡한 레이아웃

HWP는 복잡한 레이아웃을 지원하므로:
- 순수 텍스트만 추출하면 레이아웃 정보 손실
- 테이블, 이미지, 도형 등은 별도 처리 필요
- 페이지 구조가 명확하지 않을 수 있음


## 핵심 정리 및 마무리

### HWP 처리 방법 요약

| 방법 | 권장 상황 | 설치 |
|------|----------|------|
| `UnstructuredFileLoader` | 일반적인 경우 (권장) | `pip install unstructured` |
| 커스텀 로더 (`olefile`) | 세밀한 제어가 필요할 때 | `pip install olefile` |
| HWP → PDF 변환 후 처리 | 가장 확실한 방법 | 별도 변환 도구 필요 |

### 주의 사항
> HWP 파일은 **인코딩 문제**가 자주 발생해요. `cp949`, `euc-kr`, `utf-16-le` 순서로 시도해 보세요. 커스텀 로더에서 `errors='ignore'`를 사용하면 일부 문자가 손실될 수 있으니 결과를 꼭 검증해야 해요.

---

## 다음 예고

다음 노트북에서는 Office 문서 형식을 이어서 다뤄볼게요.

- **03-CSV-Loader**: 표 형식 데이터 로딩
- **04-Excel-Loader**: Excel 파일 로딩
- **05-Word-Loader**: Word 문서 로딩
- **06-PowerPoint-Loader**: 프레젠테이션 파일 로딩
